# Representation Compare Notebook

Live notebook workflow for `repr_analysis`.

This template does not require a precomputed batch directory. It can:
- run `analyze_repr.run_analysis()` directly for several checkpoints
- optionally save each per-model analysis directory
- build a compact key-metric table in memory
- save CSV / HTML tables and plot charts inline
- render PCA or t-SNE projection grids directly from the returned tensors

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

REPO_ROOT = Path.cwd()
for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (candidate / 'tools' / 'repr_analysis' / 'notebook_compare.py').exists():
        REPO_ROOT = candidate
        break

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tools.repr_analysis.notebook_compare import (
    DEFAULT_KEY_METRICS,
    build_metric_reference_table,
    build_key_metric_long_table_from_records,
    build_key_metric_table_from_records,
    plot_metric_bars,
    plot_metric_heatmap,
    plot_projection_grid_from_records,
    run_notebook_batch,
    save_metric_reference_table,
    save_key_metric_tables,
)

In [ ]:
# Edit these values for your current experiment set.
DATASET = '/opt/huawei/explorer-env/dataset/ag_data/data/world_model/quentinll/lewm-pusht/pusht_expert_train'
STATE_KEY = 'proprio'
DEVICE = 'cuda'

MODEL_SPECS = [
    ('SIGReg', '/opt/huawei/explorer-env/dataset/ag_data/data/world_model/quentinll/lewm-pusht/ckpt/pusht_lewm_20260416/pusht_lewm_20260416_epoch_9_object.ckpt'),
    ('mlp_bn_uniform_w_0p1_t_2', '/opt/huawei/explorer-env/dataset/ag_data/data/world_model/quentinll/lewm-pusht/ckpt/pusht_swm_mlp_bn_uniform_w_0p1_t_2_dim_192_20260418/pusht_swm_mlp_bn_uniform_w_0p1_t_2_dim_192_20260418_epoch_9_object.ckpt'),
]

RUN_DIR = Path('/opt/huawei/explorer-env/dataset/ag_data/data/world_model/quentinll/lewm-pusht/repr_analysis/notebook_live_compare')
ANALYSIS_SAVE_DIR = RUN_DIR / 'analysis_runs'
EXPORT_DIR = RUN_DIR / 'exports'

EVAL_SCORES = {
    'SIGReg': 82,
    'mlp_bn_uniform_w_0p1_t_2': 70,
}

MODEL_ORDER = [label for label, _ in MODEL_SPECS]
METRICS = DEFAULT_KEY_METRICS

# Live analysis settings.
ANALYZE_KWARGS = dict(
    state_key=STATE_KEY,
    frameskip=5,
    img_size=224,
    n_sequences=128,
    max_points=512,
    knn_k=10,
    action_trials=8,
    interp_steps=9,
    perturb_scale=1.0,
    seed=3072,
    device=DEVICE,
    export_tsne=False,
    tsne_perplexity=30.0,
)

In [ ]:
records = run_notebook_batch(
    MODEL_SPECS,
    dataset=DATASET,
    save_dir=ANALYSIS_SAVE_DIR,
    **ANALYZE_KWARGS,
)

print(records[0]['report'])

In [ ]:
key_wide = build_key_metric_table_from_records(
    records,
    metrics=METRICS,
    eval_scores=EVAL_SCORES,
    model_order=MODEL_ORDER,
    include_analysis_dir=True,
)

key_long = build_key_metric_long_table_from_records(
    records,
    metrics=METRICS,
    eval_scores=EVAL_SCORES,
    model_order=MODEL_ORDER,
)

display(key_wide)
save_paths = save_key_metric_tables(key_wide, key_long, output_dir=EXPORT_DIR, stem='key_metrics')
save_paths

In [ ]:
metric_reference = build_metric_reference_table(METRICS)
display(metric_reference)
reference_paths = save_metric_reference_table(metric_reference, output_dir=EXPORT_DIR)
reference_paths

In [ ]:
plot_metric_bars(
    key_wide,
    metrics=METRICS,
    output=EXPORT_DIR / 'key_metrics_bars.png',
    ncols=3,
);

In [ ]:
plot_metric_heatmap(
    key_wide,
    metrics=METRICS,
    output=EXPORT_DIR / 'key_metrics_heatmap.png',
);

In [ ]:
plot_projection_grid_from_records(
    records,
    model_labels=MODEL_ORDER,
    projection='pca',
    color_dims=[0, 1],
    output=EXPORT_DIR / 'pca_projection_grid.png',
);

If you already have a `batch_repr_analysis.py` output directory and only want to read it back, the same helper module still supports that older workflow through `build_key_metric_table(...)`, `build_key_metric_long_table(...)`, and `plot_projection_grid(...)`.